# Module 02: NLS Optimization for MIDAS

## Learning Objectives
By completing this notebook, you will:
1. Implement the profile NLS estimator for MIDAS (2D optimization over theta)
2. Visualize the profile SSE surface and understand the optimization landscape
3. Apply convergence diagnostics to verify NLS estimates
4. Compute numerical standard errors from the Hessian

## Prerequisites
- Module 01 notebooks completed
- Guide 01: NLS estimation

## Estimated Time: 15 minutes

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution
from scipy.stats import beta as beta_dist, f as f_dist
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RESOURCES_DIR = '../../module_00_foundations/resources'
if not os.path.exists(RESOURCES_DIR):
    RESOURCES_DIR = '../../../module_00_foundations/resources'

print("Imports complete.")

In [ ]:
section_divider("1. Load Data and Build MIDAS Matrix")

In [ ]:
learning_objectives(["Implement the profile NLS estimator for MIDAS (2D optimization over theta)", "Visualize the profile SSE surface and understand the optimization landscape", "Apply convergence diagnostics to verify NLS estimates", "Compute numerical standard errors from the Hessian"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load Data and Build MIDAS Matrix

In [ ]:
# Load quarterly GDP growth and monthly IP growth
gdp = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'gdp_quarterly.csv'),
    index_col='date', parse_dates=True
).squeeze()
gdp.index = pd.PeriodIndex(gdp.index, freq='Q')

ip = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'industrial_production_monthly.csv'),
    index_col='date', parse_dates=True
).squeeze()
ip.index = pd.PeriodIndex(ip.index, freq='M')

# Build MIDAS matrix: K = 12 (4 quarterly lags × 3 months)
def build_midas_matrix(y_q, x_m, n_q=4, m=3):
    K = n_q * m
    T = len(y_q)
    X = np.full((T, K), np.nan)
    for t, q in enumerate(y_q.index):
        lm = q.end_time.to_period('M')
        for j in range(K):
            tgt = lm - j
            if tgt in x_m.index:
                X[t, j] = x_m.iloc[x_m.index.get_loc(tgt)]
    Y = y_q.values
    ok = ~(np.isnan(X).any(1) | np.isnan(Y))
    return Y[ok], X[ok], y_q.index[ok]

def beta_weights(K, t1, t2):
    if t1 <= 0 or t2 <= 0: return np.ones(K)/K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, t1, t2)
    s = raw.sum()
    return raw/s if s > 1e-12 else np.ones(K)/K

Y, X, dates = build_midas_matrix(gdp, ip)
K, T = X.shape[1], len(Y)

print(f"Data ready: T={T}, K={K}")
print(f"Sample: {dates[0]} to {dates[-1]}")

In [ ]:
section_divider("2. Profile NLS: The Key Algorithm")

## 2. Profile NLS: The Key Algorithm

The profile approach reduces 4D NLS to 2D by solving analytically for $(α, β)$ at each $(θ_1, θ_2)$ evaluation.

In [ ]:
def profile_sse(theta, Y, X):
    """
    Profile SSE: for given theta, solve (alpha, beta) analytically.
    Reduces 4D optimization to 2D search over theta.
    """
    t1, t2 = theta
    if t1 <= 0 or t2 <= 0:
        return 1e10
    
    # Compute weighted aggregate
    w = beta_weights(X.shape[1], t1, t2)
    x_w = X @ w
    
    # Analytical OLS: beta = cov(y, xw) / var(xw)
    x_c = x_w - x_w.mean()
    y_c = Y - Y.mean()
    ss_x = np.dot(x_c, x_c)
    
    if ss_x < 1e-12:
        return np.sum((Y - Y.mean())**2)  # No variation in x → total SS
    
    beta = np.dot(y_c, x_c) / ss_x
    alpha = Y.mean() - beta * x_w.mean()
    
    residuals = Y - alpha - beta * x_w
    return np.sum(residuals**2)


# Visualize the 2D profile SSE landscape
theta1_grid = np.linspace(0.2, 4.0, 20)
theta2_grid = np.linspace(0.2, 10.0, 25)

profile_surface = np.zeros((len(theta1_grid), len(theta2_grid)))
for i, t1 in enumerate(theta1_grid):
    for j, t2 in enumerate(theta2_grid):
        profile_surface[i, j] = profile_sse([t1, t2], Y, X)

T1, T2 = np.meshgrid(theta1_grid, theta2_grid, indexing='ij')
best_grid = np.unravel_index(profile_surface.argmin(), profile_surface.shape)
t1_grid_opt = theta1_grid[best_grid[0]]
t2_grid_opt = theta2_grid[best_grid[1]]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Profile SSE Landscape: 2D Optimization Space\n'
             '(alpha, beta solved analytically at each theta point)', fontsize=11)

# Contour
ax1 = axes[0]
cf = ax1.contourf(T1, T2, profile_surface, levels=20, cmap='RdYlGn_r')
plt.colorbar(cf, ax=ax1, label='Profile SSE')
ax1.scatter(t1_grid_opt, t2_grid_opt, color='blue', marker='*', s=300,
            label=f'Grid min: ({t1_grid_opt:.1f}, {t2_grid_opt:.1f})', zorder=5)
ax1.scatter(1.0, 1.0, color='red', marker='o', s=100,
            label='Equal-weight (1,1)', zorder=5)
ax1.set_xlabel('θ₁'); ax1.set_ylabel('θ₂')
ax1.set_title('Profile SSE Contour Map')
ax1.legend(fontsize=8)

# SSE cross-section at grid optimal t1
ax2 = axes[1]
sse_t2 = [profile_sse([t1_grid_opt, t2], Y, X) for t2 in theta2_grid]
ax2.plot(theta2_grid, sse_t2, 'steelblue', linewidth=2.5)
ax2.axvline(t2_grid_opt, color='red', linestyle='--', label=f'Min at θ₂={t2_grid_opt:.1f}')
ax2.axvline(1.0, color='gray', linestyle=':', label='θ₂=1 (uniform)')
ax2.set_xlabel('θ₂'); ax2.set_ylabel('Profile SSE')
ax2.set_title(f'Cross-section at θ₁={t1_grid_opt:.1f}')
ax2.legend()

plt.tight_layout()
plt.savefig('profile_sse.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Profile SSE minimum at grid: θ=({t1_grid_opt:.2f}, {t2_grid_opt:.2f})")

In [ ]:
section_divider("3. Fine Optimization from Grid Starting Point")

## 3. Fine Optimization from Grid Starting Point

In [ ]:
# Fine NLS from grid minimum
result = minimize(
    profile_sse,
    [t1_grid_opt, t2_grid_opt],
    args=(Y, X),
    method='Nelder-Mead',
    options={'maxiter': 30000, 'xatol': 1e-9, 'fatol': 1e-9, 'adaptive': True}
)

t1_hat, t2_hat = result.x
t1_hat = max(t1_hat, 0.01)
t2_hat = max(t2_hat, 0.01)

# Recover (alpha, beta) by OLS at optimal theta
w_hat = beta_weights(K, t1_hat, t2_hat)
x_tilde = X @ w_hat
ols = LinearRegression().fit(x_tilde.reshape(-1,1), Y)
alpha_hat = ols.intercept_
beta_hat = ols.coef_[0]
fitted = ols.predict(x_tilde.reshape(-1,1))
residuals = Y - fitted
r2 = r2_score(Y, fitted)

print("Profile NLS Results:")
print(f"  theta1 = {t1_hat:.4f}")
print(f"  theta2 = {t2_hat:.4f}")
print(f"  alpha  = {alpha_hat:.4f}")
print(f"  beta   = {beta_hat:.4f}")
print(f"  R²     = {r2:.4f}")
print(f"  Converged: {result.success}")
print(f"  Profile SSE: {result.fun:.4f}")

In [ ]:
section_divider("4. HAC Standard Errors")

## 4. HAC Standard Errors

In [ ]:
# HAC standard errors for (alpha, beta)
# Linearized model: Y = alpha + beta * x_tilde + residuals
L_nw = int(4 * (T/100)**(2/9))  # Newey-West bandwidth

Z = sm.add_constant(x_tilde)
hac_model = sm.OLS(Y, Z)
hac_result = hac_model.fit(cov_type='HAC', cov_kwds={'maxlags': L_nw})

print(f"HAC Standard Errors (Newey-West, L={L_nw} lags)")
print("=" * 55)
print(hac_result.summary().tables[1])
print(f"\nR² = {hac_result.rsquared:.4f}")
print(f"F-statistic: {hac_result.fvalue:.3f} (p={hac_result.f_pvalue:.4f})")

In [ ]:
section_divider("5. Test Equal-Weight Restriction")

## 5. Test Equal-Weight Restriction

In [ ]:
# F-test: H0: Beta(1,1) = equal-weight aggregation

# Restricted model: equal-weight OLS
x_equal = X.mean(axis=1)  # Equal-weight aggregate
ols_r = LinearRegression().fit(x_equal.reshape(-1,1), Y)
sse_restricted = np.sum((Y - ols_r.predict(x_equal.reshape(-1,1)))**2)

# Unrestricted model: profile NLS
sse_unrestricted = result.fun

r_restrictions = 2  # theta1 = 1, theta2 = 1
k_unrestricted = 4  # alpha, beta, theta1, theta2
df1 = r_restrictions
df2 = T - k_unrestricted

F_stat = ((sse_restricted - sse_unrestricted) / df1) / (sse_unrestricted / df2)
p_value = 1 - f_dist.cdf(F_stat, df1, df2)

print("F-test: H0 = Equal-Weight Aggregation (Beta(1,1))")
print("-" * 50)
print(f"SSE restricted (OLS-aggregate): {sse_restricted:.4f}")
print(f"SSE unrestricted (MIDAS):        {sse_unrestricted:.4f}")
print(f"SSE reduction:                   {sse_restricted - sse_unrestricted:.4f} ({(sse_restricted-sse_unrestricted)/sse_restricted:.1%})")
print(f"F({df1}, {df2}) = {F_stat:.4f}")
print(f"p-value = {p_value:.4f}")
if p_value < 0.05:
    print("\nConclusion: REJECT H0 at 5% — MIDAS weights differ significantly from uniform.")
    print("The polynomial restriction provides statistically significant improvement over aggregation.")
else:
    print("\nConclusion: FAIL TO REJECT H0 — equal-weight aggregation is adequate.")

## 6. Residual Diagnostics

In [ ]:
from scipy.stats import chi2

# Ljung-Box test for serial correlation
def ljung_box(resid, n_lags=4):
    n = len(resid)
    acf = [np.corrcoef(resid[k:], resid[:-k])[0,1] for k in range(1, n_lags+1)]
    Q = n * (n+2) * sum(r**2/(n-k) for k, r in enumerate(acf, 1))
    p_val = 1 - chi2.cdf(Q, df=n_lags)
    return Q, p_val, acf

Q, p_lb, acf_vals = ljung_box(residuals, n_lags=4)

print("Residual Diagnostics")
print("=" * 40)
print(f"\nLjung-Box Q(4) = {Q:.3f}, p-value = {p_lb:.4f}")
if p_lb < 0.10:
    print("  → Reject H0: evidence of serial correlation. Consider adding AR(1) term.")
else:
    print("  → No significant serial correlation.")

print(f"\nACF of residuals:")
for k, rho in enumerate(acf_vals, 1):
    bar = '|' + '#' * int(abs(rho) * 30) + ' ' * (30 - int(abs(rho)*30)) + '|'
    sign = '+' if rho >= 0 else '-'
    print(f"  lag {k}: {sign}{bar} {abs(rho):.3f}")

# Plot residuals
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
t_axis = range(len(residuals))

ax1 = axes[0]
ax1.plot(t_axis, residuals, 'steelblue', linewidth=1)
ax1.axhline(0, color='black', linewidth=0.5)
ax1.axhline(2*residuals.std(), color='red', linestyle='--', alpha=0.5)
ax1.axhline(-2*residuals.std(), color='red', linestyle='--', alpha=0.5)
ax1.set_title('Residuals over time')
ax1.set_ylabel('Residual (%)')

ax2 = axes[1]
ax2.hist(residuals, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
ax2.set_title('Residual distribution')
ax2.set_xlabel('Residual (%)')

ax3 = axes[2]
max_lag_plot = 8
acf_extended = [np.corrcoef(residuals[k:], residuals[:-k])[0,1]
                for k in range(1, max_lag_plot+1)]
ax3.bar(range(1, max_lag_plot+1), acf_extended, color='steelblue', alpha=0.8)
ax3.axhline(0, color='black', linewidth=0.5)
ax3.axhline(1.96/np.sqrt(T), color='red', linestyle='--', alpha=0.6, label='95% bands')
ax3.axhline(-1.96/np.sqrt(T), color='red', linestyle='--', alpha=0.6)
ax3.set_xlabel('Lag'); ax3.set_ylabel('ACF')
ax3.set_title('Residual ACF')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('residual_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

### What You Built
1. **Profile NLS**: 2D optimization over $(\theta_1, \theta_2)$ with $(\alpha, \beta)$ solved analytically
2. **Convergence diagnostics**: SSE landscape visualization, gradient check
3. **HAC standard errors**: Newey-West correction for serial correlation and heteroscedasticity
4. **F-test**: Formal test of equal-weight restriction
5. **Residual diagnostics**: Ljung-Box and ACF plots

### Key Results
- Profile NLS is more reliable than joint 4D optimization for MIDAS
- HAC standard errors are wider than OLS standard errors (correct for serial correlation)
- The F-test result determines whether MIDAS adds value over simple aggregation
- Residual ACF shows whether AR terms are needed

---
*Next: Module 02, Notebook 02 — Model Selection (AIC/BIC, expanding window CV)*

In [ ]:
key_takeaways(["Profile NLS", "Convergence diagnostics", "HAC standard errors", "F-test", "Residual diagnostics"])